In [2]:
!pip install requests
!pip install datetime

import requests
import pandas as pd
from datetime import date, timedelta


  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
Using cached pytz-2026.1.post1-py2.py3-none-any.whl (510 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [datetime]1/3 [zope.interface]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [63]:
import requests
import pandas as pd

LAT, LON = 56.1629, 10.2039  # Aarhus
START, END = "2022-01-01", "2026-04-18"

# ── 1. Historical air quality (NO2, O3) ──────────────────────────────────────
print("Fetching air quality...")
aq = requests.get(
    "https://air-quality-api.open-meteo.com/v1/air-quality",
    params={
        "latitude": LAT,
        "longitude": LON,
        "hourly": "nitrogen_dioxide,ozone,pm10,pm2_5",
        "start_date": START,
        "end_date": END,
        "timezone": "Europe/Copenhagen"
    }
).json()

aq_df = pd.DataFrame({
    "Recorded": pd.to_datetime(aq["hourly"]["time"]),
    "NO2":      aq["hourly"]["nitrogen_dioxide"],
    "O3":       aq["hourly"]["ozone"],
    "PM10":     aq["hourly"]["pm10"],
    "PM2.5":    aq["hourly"]["pm2_5"],
})

# ── 2. Historical weather (wind, rain, temp, radiation) ──────────────────────
print("Fetching weather...")
wx = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": LAT,
        "longitude": LON,
        "hourly": "wind_speed_10m,wind_direction_10m,temperature_2m,shortwave_radiation,precipitation,relative_humidity_2m",
        "wind_speed_unit": "ms",
        "start_date": START,
        "end_date": END,
        "timezone": "Europe/Copenhagen"
    }
).json()

wx_df = pd.DataFrame({
    "Recorded":           pd.to_datetime(wx["hourly"]["time"]),
    "wind_speed":         wx["hourly"]["wind_speed_10m"],
    "wind_direction":     wx["hourly"]["wind_direction_10m"],
    "temperature":        wx["hourly"]["temperature_2m"],
    "solar_radiation":    wx["hourly"]["shortwave_radiation"],
    "precipitation":      wx["hourly"]["precipitation"],
    "humidity":           wx["hourly"]["relative_humidity_2m"],

})

# ── 3. Merge and save ─────────────────────────────────────────────────────────
print("Merging...")
df = aq_df.merge(wx_df, on="Recorded", how="inner")
df = df.sort_values("Recorded").reset_index(drop=True)

print(f"\nFinal shape: {df.shape}")
print(df.head())
print(f"\nDate range: {df.Recorded.min()} → {df.Recorded.max()}")
print(f"Missing values:\n{df.isnull().sum()}")

df.to_csv("aarhus_air_quality.csv", index=False)
print("\nSaved to aarhus_air_quality.csv")

Fetching air quality...
Fetching weather...
Merging...

Final shape: (37656, 11)
             Recorded  NO2    O3  PM10  PM2.5  wind_speed  wind_direction  \
0 2022-01-01 00:00:00  6.2  53.0   6.2    5.0        5.00             269   
1 2022-01-01 01:00:00  4.6  53.0   6.4    4.7        5.00             272   
2 2022-01-01 02:00:00  5.0  57.0   6.0    4.5        4.90             269   
3 2022-01-01 03:00:00  5.0  54.0   5.4    4.6        4.72             265   
4 2022-01-01 04:00:00  5.7  52.0   5.3    4.9        4.53             264   

   temperature  solar_radiation  precipitation  humidity  
0          6.7              0.0            0.0        98  
1          7.2              0.0            0.0        98  
2          7.1              0.0            0.0        97  
3          7.1              0.0            0.0        97  
4          7.1              0.0            0.0        96  

Date range: 2022-01-01 00:00:00 → 2026-04-18 23:00:00
Missing values:
Recorded           0
NO2       

In [64]:
df.head()

,Recorded,NO2,O3,PM10,PM2.5,wind_speed,wind_direction,temperature,solar_radiation,precipitation,humidity
0,2022-01-01 00:00:00,6.2,53.0,6.2,5.0,5.00,269,6.7,0.0,0.0,98
1,2022-01-01 01:00:00,4.6,53.0,6.4,4.7,5.00,272,7.2,0.0,0.0,98
2,2022-01-01 02:00:00,5.0,57.0,6.0,4.5,4.90,269,7.1,0.0,0.0,97
3,2022-01-01 03:00:00,5.0,54.0,5.4,4.6,4.72,265,7.1,0.0,0.0,97
4,2022-01-01 04:00:00,5.7,52.0,5.3,4.9,4.53,264,7.1,0.0,0.0,96


In [65]:
df.describe()

,Recorded,NO2,O3,PM10,PM2.5,wind_speed,wind_direction,temperature,solar_radiation,precipitation,humidity
count,37656,37656.000000,37656.000000,37656.000000,37656.000000,37656.000000,37656.000000,37656.000000,37656.000000,37656.000000,37656.000000
mean,2024-02-24 11:30:00,6.091096,62.086908,10.321943,6.716380,4.660753,203.508631,9.057226,123.618733,0.093233,81.081820
min,2022-01-01 00:00:00,0.600000,4.000000,1.300000,0.900000,0.050000,1.000000,-12.600000,0.000000,0.000000,22.000000
25%,2023-01-28 05:45:00,3.200000,51.000000,6.800000,3.700000,3.020000,138.000000,4.200000,0.000000,0.000000,73.000000
50%,2024-02-24 11:30:00,5.000000,63.000000,9.200000,5.400000,4.430000,219.000000,8.800000,4.000000,0.000000,84.000000
75%,2025-03-22 17:15:00,7.800000,74.000000,12.500000,8.300000,6.040000,272.000000,14.100000,180.000000,0.000000,92.000000
max,2026-04-18 23:00:00,38.000000,159.000000,59.100000,46.700000,15.370000,360.000000,27.700000,864.000000,11.200000,100.000000
std,NaN,4.093075,17.348296,5.206703,4.470867,2.184536,85.405097,6.468670,196.063059,0.386187,13.083754
